In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import re

warnings.filterwarnings("ignore")

In [2]:
BASE_DIR = Path(r"C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting")

XLS_DIR = BASE_DIR / "nldc_xls"

YEAR_FOLDERS = [
    XLS_DIR / "2023-24",
    XLS_DIR / "2024-25",
    XLS_DIR / "2025-26"
]

# Output folders
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR)

Output folder: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\data\processed


In [3]:
def extract_date_from_filename(file_path):
    try:
        return pd.to_datetime(file_path.stem.split("_")[0])
    except:
        return None

## Part A: Daily Data (MOP_E)

In [4]:
def extract_daily_peak(file_path):
    try:
        df = pd.read_excel(file_path, sheet_name="MOP_E", header=None)

        for i in range(len(df)):
            row = df.iloc[i]

            row_text = " ".join(row.astype(str)).lower()

            if "maximum demand met during the day" in row_text:

                numeric_values = []

                for val in row:
                    try:
                        num = float(val)
                        if not np.isnan(num):
                            numeric_values.append(num)
                    except:
                        continue

                if len(numeric_values) == 0:
                    return None

                # ✅ FIX: take MAX value (TOTAL)
                return max(numeric_values)

        return None

    except:
        return None

In [5]:
daily_data = []

for folder in YEAR_FOLDERS:
    print(f"\nProcessing {folder}")

    for file_path in folder.rglob("*.xls*"):

        date = extract_date_from_filename(file_path)
        if date is None:
            continue

        peak = extract_daily_peak(file_path)

        if peak is not None:
            daily_data.append({
                "date": date,
                "max_demand_met_mw": peak
            })

daily_df = pd.DataFrame(daily_data)

print("Daily dataset shape:", daily_df.shape)


Processing C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\nldc_xls\2023-24

Processing C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\nldc_xls\2024-25

Processing C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\nldc_xls\2025-26
Daily dataset shape: (1086, 2)


In [6]:
import pandas as pd

bad_files = []
daily_data = []

for folder in YEAR_FOLDERS:
    for file_path in folder.rglob("*.xls*"):

        date = extract_date_from_filename(file_path)
        if date is None:
            continue

        try:
            peak = extract_daily_peak(file_path)
        except Exception:
            bad_files.append(file_path)
            continue

        # 🚨 KEY FIX: detect invalid outputs
        if peak is None or peak < 50000 or peak > 400000:
            bad_files.append(file_path)
            continue

        daily_data.append({
            "date": date,
            "max_demand_met_mw": peak
        })

print(f"\nBad/corrupted files detected: {len(bad_files)}")

# Remove duplicates in bad_files
bad_files = list(set(bad_files))

# Delete them
for f in bad_files:
    try:
        f.unlink()
    except Exception as e:
        print(f"Could not delete {f}: {e}")

print("Bad files removed.")

# Rebuild dataset
daily_df = pd.DataFrame(daily_data)
daily_df["date"] = pd.to_datetime(daily_df["date"])
daily_df = daily_df.sort_values("date")

print("\nClean dataset shape:", daily_df.shape)

# Check missing dates
missing_dates = pd.date_range(
    daily_df["date"].min(),
    daily_df["date"].max()
).difference(daily_df["date"])

print("Missing dates count:", len(missing_dates))

if len(missing_dates) > 0:
    print("Missing dates:", missing_dates)
else:
    print("Dataset is complete ✅")


Bad/corrupted files detected: 0
Bad files removed.

Clean dataset shape: (1086, 2)
Missing dates count: 11
Missing dates: DatetimeIndex(['2023-06-20', '2023-06-26', '2023-06-28', '2023-07-06',
               '2023-12-15', '2024-01-04', '2024-10-14', '2024-12-30',
               '2024-12-31', '2025-10-06', '2025-11-10'],
              dtype='datetime64[ns]', freq=None)


In [7]:
daily_df["date"] = pd.to_datetime(daily_df["date"])

daily_df["day_of_week"] = daily_df["date"].dt.dayofweek
daily_df["month"] = daily_df["date"].dt.month
daily_df["is_weekend"] = daily_df["day_of_week"].isin([5, 6]).astype(int)

In [8]:
HOLIDAYS_PATH = BASE_DIR / "holidays.csv"  # adjust if needed

holidays_df = pd.read_csv(HOLIDAYS_PATH)

holidays_df["date"] = pd.to_datetime(holidays_df["date"], dayfirst=True)

In [9]:
# Normalize columns
holidays_df.columns = holidays_df.columns.str.strip().str.lower()

# Convert date
holidays_df["date"] = pd.to_datetime(holidays_df["date"])

# Clean values
holidays_df["pre_post_flag"] = holidays_df["pre_post_flag"].astype(str).str.strip().str.lower()

# Filter only all-India holidays
holidays_df = holidays_df[holidays_df["impact_scope"] == "all_india"]

# Create features
holidays_df["is_festival"] = holidays_df["is_holiday"]
holidays_df["is_pre"] = (holidays_df["pre_post_flag"] == "pre").astype(int)
holidays_df["is_post"] = (holidays_df["pre_post_flag"] == "post").astype(int)

In [10]:
# Assign priority: festival > pre > post
def assign_priority(row):
    if row["is_festival"] == 1:
        return 3
    elif row["is_pre"] == 1:
        return 2
    elif row["is_post"] == 1:
        return 1
    else:
        return 0

holidays_df["priority"] = holidays_df.apply(assign_priority, axis=1)

# Sort and keep highest priority per date
holidays_df = holidays_df.sort_values(["date", "priority"], ascending=[True, False])

holidays_df = holidays_df.drop_duplicates(subset="date", keep="first")

# Keep only required columns
holidays_df = holidays_df[[
    "date",
    "is_festival",
    "is_pre",
    "is_post",
    "reference_festival"
]].rename(columns={"reference_festival": "festival_name"})

In [11]:
daily_df = daily_df.merge(holidays_df, on="date", how="left")

# Fill missing values
daily_df["is_festival"] = daily_df["is_festival"].fillna(0)
daily_df["is_pre"] = daily_df["is_pre"].fillna(0)
daily_df["is_post"] = daily_df["is_post"].fillna(0)
daily_df["festival_name"] = daily_df["festival_name"].fillna("none")

In [12]:
daily_df = daily_df.sort_values("date").reset_index(drop=True)

daily_path = OUTPUT_DIR / "daily_all_india_2023_26.csv"
daily_df.to_csv(daily_path, index=False)

print("Saved daily dataset:", daily_path)
print("Final shape:", daily_df.shape)

Saved daily dataset: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\data\processed\daily_all_india_2023_26.csv
Final shape: (1086, 9)


## Part B: 15-Min Data (2025-26)

In [13]:
def read_timeseries_sheet(file_path):
    try:
        xl = pd.read_excel(file_path, sheet_name=None, engine="xlrd")
    except:
        try:
            xl = pd.read_excel(file_path, sheet_name=None, engine="openpyxl")
        except:
            return None

    for sheet in xl.keys():
        if "time" in sheet.lower():
            return xl[sheet]

    return None

In [14]:
def parse_timeseries(file_path):
    df = read_timeseries_sheet(file_path)
    if df is None:
        return None

    date = extract_date_from_filename(file_path)
    if date is None:
        return None

    df.columns = df.columns.str.strip().str.lower()

    # Find time column
    time_cols = [col for col in df.columns if "time" in col]
    if len(time_cols) == 0:
        return None

    time_col = time_cols[0]

    # Create datetime
    df["datetime"] = pd.to_datetime(
        date.strftime("%Y-%m-%d") + " " + df[time_col].astype(str),
        errors="coerce"
    )

    df["date"] = df["datetime"].dt.date

    # Rename demand column
    for col in df.columns:
        if "demand" in col and "met" in col:
            df.rename(columns={col: "demand_met_mw"}, inplace=True)

    return df

In [15]:
def parse_timeseries(file_path):
    try:
        # Read raw sheet (no header assumption)
        df = pd.read_excel(file_path, sheet_name="TimeSeries", header=None)

        header_row = None

        # Step 1: Find header row dynamically
        for i in range(len(df)):
            row = df.iloc[i].astype(str).str.lower().tolist()

            if any(("time" in x) or ("block" in x) for x in row):
                header_row = i
                break

        if header_row is None:
            return None

        # Step 2: Extract header
        headers = df.iloc[header_row].astype(str).str.strip().tolist()

        # Step 3: Extract data below header
        data = df.iloc[header_row + 1:].copy()
        data.columns = headers

        # Step 4: Drop completely empty rows
        data = data.dropna(how="all")

        # Step 5: Identify columns
        time_col = data.columns[0]

        if len(data.columns) < 3:
            return None

        demand_col = data.columns[2]  # Based on NLDC structure

        # Step 6: Get date
        date = extract_date_from_filename(file_path)
        if date is None:
            return None

        # Step 7: Create datetime
        data["datetime"] = pd.to_datetime(
            date.strftime("%Y-%m-%d") + " " + data[time_col].astype(str),
            errors="coerce"
        )

        # Step 8: Extract demand
        data["demand_met_mw"] = pd.to_numeric(data[demand_col], errors="coerce")

        # Step 9: Keep only required columns
        data = data[["datetime", "demand_met_mw"]]

        return data

    except:
        return None

In [16]:
timeseries_list = []

folder_2025 = XLS_DIR / "2025-26"

files = list(folder_2025.rglob("*.xls*"))
print("Files found:", len(files))

for file_path in files:
    ts = parse_timeseries(file_path)
    if ts is not None:
        timeseries_list.append(ts)

timeseries_df = pd.concat(timeseries_list, ignore_index=True)

print("Timeseries shape:", timeseries_df.shape)

Files found: 363
Timeseries shape: (35344, 2)


In [17]:
# Remove invalid timestamps
timeseries_df = timeseries_df.dropna(subset=["datetime"])

# Sort properly
timeseries_df = timeseries_df.sort_values("datetime")

# Remove duplicates
timeseries_df = timeseries_df.drop_duplicates(subset=["datetime"])

# Reset index
timeseries_df = timeseries_df.reset_index(drop=True)

print("After cleaning:", timeseries_df.shape)

After cleaning: (34618, 2)


In [18]:
timeseries_df["date"] = timeseries_df["datetime"].dt.date
timeseries_df["hour"] = timeseries_df["datetime"].dt.hour
timeseries_df["minute"] = timeseries_df["datetime"].dt.minute
timeseries_df["day_of_week"] = timeseries_df["datetime"].dt.dayofweek

print("Feature columns added")

Feature columns added


In [19]:
print("\nHead:")
print(timeseries_df.head())

print("\nNull values:")
print(timeseries_df.isnull().sum())

print("\nDate range:")
print(timeseries_df["datetime"].min(), "→", timeseries_df["datetime"].max())


Head:
             datetime  demand_met_mw        date  hour  minute  day_of_week
0 2025-04-01 00:00:00       191368.0  2025-04-01     0       0            1
1 2025-04-01 00:15:00       190921.0  2025-04-01     0      15            1
2 2025-04-01 00:30:00       190222.0  2025-04-01     0      30            1
3 2025-04-01 00:45:00       188971.0  2025-04-01     0      45            1
4 2025-04-01 01:00:00       188162.0  2025-04-01     1       0            1

Null values:
datetime         0
demand_met_mw    0
date             0
hour             0
minute           0
day_of_week      0
dtype: int64

Date range:
2025-04-01 00:00:00 → 2026-03-31 23:45:00


In [20]:
ts_path = OUTPUT_DIR / "timeseries_15min_2025_26.csv"

timeseries_df.to_csv(ts_path, index=False)

print("Saved timeseries dataset at:", ts_path)

Saved timeseries dataset at: C:\Users\Shriyans S Sahoo\DS4SG\Electricity-Demand-Forecasting\data\processed\timeseries_15min_2025_26.csv


## Data Overview

In [21]:
timeseries_df.head()


,datetime,demand_met_mw,date,hour,minute,day_of_week
0,2025-04-01 00:00:00,191368.0,2025-04-01,0,0,1
1,2025-04-01 00:15:00,190921.0,2025-04-01,0,15,1
2,2025-04-01 00:30:00,190222.0,2025-04-01,0,30,1
3,2025-04-01 00:45:00,188971.0,2025-04-01,0,45,1
4,2025-04-01 01:00:00,188162.0,2025-04-01,1,0,1


In [22]:
timeseries_df.describe()


,datetime,demand_met_mw,hour,minute,day_of_week
count,34618,34618.000000,34618.000000,34618.000000,34618.000000
mean,2025-09-30 16:41:41.660985600,195634.335866,11.491941,22.497400,3.019412
min,2025-04-01 00:00:00,128473.000000,0.000000,0.000000,0.000000
25%,2025-06-30 04:18:45,183984.000000,5.000000,0.000000,1.000000
50%,2025-09-30 07:52:30,199107.000000,11.000000,15.000000,3.000000
75%,2025-12-31 19:56:15,210895.000000,17.000000,30.000000,5.000000
max,2026-03-31 23:45:00,245068.000000,23.000000,45.000000,6.000000
std,NaN,21084.572123,6.921212,16.770364,1.995944


In [23]:
timeseries_df.isnull().sum()

datetime         0
demand_met_mw    0
date             0
hour             0
minute           0
day_of_week      0
dtype: int64

In [24]:
daily_df.head()

,date,max_demand_met_mw,day_of_week,month,is_weekend,is_festival,is_pre,is_post,festival_name
0,2023-04-01,180284.0,5,4,1,0.0,0.0,1.0,Ram Navami
1,2023-04-02,174691.0,6,4,1,0.0,0.0,0.0,none
2,2023-04-03,187729.0,0,4,0,0.0,1.0,0.0,Mahavir Jayanti
3,2023-04-04,189684.0,1,4,0,1.0,0.0,0.0,none
4,2023-04-05,189003.0,2,4,0,0.0,1.0,0.0,Hanuman Jayanti


In [25]:
daily_df.describe()

,date,max_demand_met_mw,day_of_week,month,is_weekend,is_festival,is_pre,is_post
count,1086,1086.000000,1086.000000,1086.000000,1086.000000,1086.000000,1086.000000,1086.000000
mean,2024-10-01 06:45:44.751381248,213388.145552,3.015654,6.498158,0.289134,0.097606,0.116022,0.102210
min,2023-04-01 00:00:00,159720.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,2024-01-02 06:00:00,204563.500000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000
50%,2024-09-30 12:00:00,214075.500000,3.000000,7.000000,0.000000,0.000000,0.000000,0.000000
75%,2025-07-01 18:00:00,223103.500000,5.000000,9.000000,1.000000,0.000000,0.000000,0.000000
max,2026-03-31 00:00:00,250070.000000,6.000000,12.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,14636.111526,2.000630,3.446964,0.453569,0.296918,0.320399,0.303064


In [26]:
daily_df.isnull().sum()

date                 0
max_demand_met_mw    0
day_of_week          0
month                0
is_weekend           0
is_festival          0
is_pre               0
is_post              0
festival_name        0
dtype: int64

In [27]:
file = list((XLS_DIR / "2023-24").rglob("*.xls*"))[0]

df = pd.read_excel(file, sheet_name="MOP_E", header=None)

for i in range(len(df)):
    row = df.iloc[i].astype(str).str.lower().tolist()
    
    if "maximum demand met during the day" in " ".join(row):
        print("FOUND ROW:", i)
        print(df.iloc[i])
        break

FOUND ROW: 12
0    Maximum Demand Met During the Day (MW) (From N...
1                                                  NaN
2                                                45142
3                                                59789
4                                                60780
5                                                20186
6                                                 2557
7                                               180284
8                                                  NaN
Name: 12, dtype: object
